# Notebook 07 — Shrink and Quantize (The Hardware-Bound Model)

*The final PyTorch notebook. Output: an int8 weight file the 6502 can read out of EEPROM.*

## What we're doing and why

This is the production model. Every prior notebook was a learning step; this one produces the artifact.

### Steps

1. **Train** a hardware-spec model in fp32 (d=8, ctx=16, 1 head, 1 layer, vocab=32). Same architecture as nb05's 1-layer.
2. **Quantize** all weights to **int8** (one byte per weight). Discuss why activations stay int16.
3. **Evaluate** the quantized model — how much loss did we lose?
4. **Pack** into a binary file with a header describing layout, scales, and zero-points.
5. **Estimate** cycles per token and confirm we're within the ~150 ms budget at 1 MHz.

After this notebook, the rest of the project is C and assembly.

### Why int8?

The math: 8 KB EEPROM, ~830 model params at fp32 = ~3.3 KB just for weights — fits, but leaves no room for the tokenizer table, vocabulary, or program code. At **int8**, weights are ~830 bytes. The remaining ~7 KB holds program, vocab table, and constants.

### Why activations stay int16 (not int8)

Inside attention you do `Q @ K^T`, a sum of `head_size=8` products. Each product is `int8 × int8 = int16`. Sum of 8 of those fits comfortably in int16 (max value ≈ 8 × 127² = 129,000 → need int24, but in practice values are nowhere near max; we'll add saturation logic in the C reference). Squeezing accumulators into int8 would force aggressive scaling and cost us most of the model's remaining quality.

This is **standard MCU LLM practice**: int8 weights, wider accumulators. TFLite Micro does the same.


## Cell 1 — Setup + train the hardware-spec model (fp32)

Reuse the architecture from nb05 (1-layer pre-norm block with MLP). Train it. We'll quantize this checkpoint.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, struct
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

text = Path('../data/tinyshakespeare.txt').read_text().lower()
VOCAB_SIZE = 32
top = [c for c, _ in Counter(text).most_common(VOCAB_SIZE - 1)]
itos = ['<unk>'] + top
stoi = {c: i for i, c in enumerate(itos)}
encode = lambda s: [stoi.get(c, 0) for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

BATCH_SIZE=64; BLOCK_SIZE=16; EMBED_DIM=8; NUM_HEADS=1; N_LAYERS=1
LR=3e-3; N_STEPS=8000; EVAL_EVERY=500

def get_batch(split):
    d = train_data if split=='train' else val_data
    ix = torch.randint(0, len(d)-BLOCK_SIZE-1, (BATCH_SIZE,))
    x = torch.stack([d[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)

# --- Architecture (copied from nb05) ---
class Head(nn.Module):
    def __init__(self, embed_dim, head_size, block_size):
        super().__init__()
        self.key   = nn.Linear(embed_dim, head_size, bias=False)
        self.query = nn.Linear(embed_dim, head_size, bias=False)
        self.value = nn.Linear(embed_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size
    def forward(self, x):
        B, T, _ = x.shape
        k=self.key(x); q=self.query(x); v=self.value(x)
        s = q @ k.transpose(-2, -1) / (self.head_size**0.5)
        s = s.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        return F.softmax(s, dim=-1) @ v

class MultiHead(nn.Module):
    def __init__(self, ed, nh, bs):
        super().__init__()
        self.heads = nn.ModuleList([Head(ed, ed//nh, bs) for _ in range(nh)])
        self.proj  = nn.Linear(ed, ed)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FF(nn.Module):
    def __init__(self, ed, mult=4):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(ed, mult*ed), nn.ReLU(), nn.Linear(mult*ed, ed))
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, ed, nh, bs):
        super().__init__()
        self.ln1=nn.LayerNorm(ed); self.attn=MultiHead(ed,nh,bs)
        self.ln2=nn.LayerNorm(ed); self.mlp=FF(ed)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyTransformer(nn.Module):
    def __init__(self, V, ed, nh, bs, L):
        super().__init__()
        self.block_size = bs
        self.token_embed = nn.Embedding(V, ed)
        self.pos_embed   = nn.Embedding(bs, ed)
        self.blocks = nn.Sequential(*[Block(ed, nh, bs) for _ in range(L)])
        self.ln_final = nn.LayerNorm(ed)
        self.lm_head  = nn.Linear(ed, V)
    def forward(self, idx, targets=None):
        B,T = idx.shape
        x = self.token_embed(idx) + self.pos_embed(torch.arange(T, device=idx.device))
        x = self.blocks(x); x = self.ln_final(x)
        logits = self.lm_head(x)
        if targets is None: return logits, None
        loss = F.cross_entropy(logits.view(B*T,-1), targets.view(B*T))
        return logits, loss

model = TinyTransformer(VOCAB_SIZE, EMBED_DIM, NUM_HEADS, BLOCK_SIZE, N_LAYERS).to(device)
print(f'parameters: {sum(p.numel() for p in model.parameters())}')


In [ ]:
# Train
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
history = []
for step in range(N_STEPS + 1):
    if step % EVAL_EVERY == 0:
        model.eval()
        with torch.no_grad():
            ls = {}
            for split in ('train','val'):
                a = torch.zeros(20)
                for k in range(20):
                    xb,yb = get_batch(split); _,l = model(xb,yb); a[k]=l.item()
                ls[split] = a.mean().item()
        history.append((step, ls['train'], ls['val']))
        print(f'step {step:>5} | train {ls["train"]:.4f} | val {ls["val"]:.4f}')
        model.train()
    xb,yb = get_batch('train')
    _,loss = model(xb,yb)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

val_fp32 = history[-1][2]
print(f'\nfp32 baseline val loss: {val_fp32:.4f}')


## Cell 2 — Post-training quantization (PTQ), symmetric per-tensor int8

### The math of int8 quantization

For a float tensor `W`, we want to represent each value with an 8-bit signed integer ∈ [-128, 127]. We need a **scale** `s` and (optionally) a **zero point** `z` to map back to floats:

$$w_{\text{float}} = s \cdot (w_{\text{int}} - z)$$

We'll use **symmetric quantization** (`z = 0`), which is simpler — the int8 zero corresponds to float zero. Asymmetric (with z ≠ 0) packs slightly more range but adds an extra subtraction in inference. Not worth it on a 6502.

Per-tensor scale:
$$s = \frac{\max(|W|)}{127}$$

Then `w_int = round(w_float / s).clip(-128, 127)`.

### Per-tensor vs per-channel

- **Per-tensor**: one scale for the whole matrix. Simplest. We use this.
- **Per-channel** (per output row): one scale per row. More accurate at the cost of a bigger scale table. Standard in modern QAT.

At our scale, per-tensor is fine and saves bytes.

### Why post-training (PTQ) not quantization-aware (QAT)?

- **PTQ**: train fp32, then convert. Simple, fast, usually loses 1-5% accuracy.
- **QAT**: simulate int8 *during* training so the model adapts. More accurate, more complex.

For 832 params and a tiny vocab, PTQ loss is small and the code is dramatically simpler. We're aiming for **<10% loss degradation**.


In [ ]:
def quantize_symmetric(t: torch.Tensor):
    '''Symmetric per-tensor int8 quantization. Returns (int8_tensor, scale).'''
    max_abs = t.abs().max().item()
    if max_abs == 0:
        return torch.zeros_like(t, dtype=torch.int8), 1.0
    scale = max_abs / 127.0
    qt = torch.round(t / scale).clamp(-128, 127).to(torch.int8)
    return qt, scale

def dequantize(qt: torch.Tensor, scale: float):
    return qt.to(torch.float32) * scale

# Demo on the lm_head weights
W = model.lm_head.weight.data
qW, s = quantize_symmetric(W)
W_rec = dequantize(qW, s)
err = (W - W_rec).abs().max().item()
print(f'lm_head.weight: shape={tuple(W.shape)}  range=[{W.min():.3f}, {W.max():.3f}]')
print(f'  scale={s:.6f}  max reconstruction error={err:.6f}')


## Cell 3 — Quantize the whole model and evaluate

Walk every weight tensor, quantize it, store the dequantized version back into the model. This **simulates** what int8 inference would compute (the model still does fp32 math at runtime, but the values it multiplies are restricted to those expressible in int8). It's the cheapest way to measure quantization-induced loss without writing an int8 inference engine in Python.

We **skip** LayerNorm parameters — γ and β need fp range to work properly. On the 6502 we'll either keep these in higher precision (e.g. int16 with a per-channel scale) or absorb them into surrounding operations. We'll handle that in the C reference.


In [ ]:
quant_table = {}

for name, p in model.named_parameters():
    if 'ln' in name or 'norm' in name:
        continue  # keep LayerNorm in fp
    if p.dim() < 1:
        continue
    qp, s = quantize_symmetric(p.data)
    quant_table[name] = (qp, s)
    p.data.copy_(dequantize(qp, s))

# Re-evaluate
model.eval()
with torch.no_grad():
    losses = torch.zeros(40)
    for k in range(40):
        xb, yb = get_batch('val'); _, l = model(xb, yb); losses[k] = l.item()
    val_int8 = losses.mean().item()

print(f'fp32 val loss:  {val_fp32:.4f}')
print(f'int8 val loss:  {val_int8:.4f}')
print(f'degradation:    {(val_int8 - val_fp32):+.4f} nats  ({100*(val_int8/val_fp32 - 1):+.2f}%)')

print('\nquantized tensors:')
for name, (qp, s) in quant_table.items():
    print(f'  {name:35s} {tuple(qp.shape)}  scale={s:.6f}')


## Cell 4 — Generate from the quantized model

Sanity check: does the int8 model still produce text in the same style as the fp32 one? Should be qualitatively identical — same gibberish-shape, similar word-ish fragments.


In [ ]:
@torch.no_grad()
def generate(model, prompt='\n', max_new_tokens=400):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -BLOCK_SIZE:]
        logits, _ = model(idx_cond)
        probs = F.softmax(logits[:, -1, :], dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, nxt], dim=1)
    return decode(idx[0].tolist())

print(generate(model))


## Cell 5 — Pack the int8 weights into a binary EEPROM image

### File format (`wozformer.bin`)

A simple flat binary, little-endian. Designed so the 6502 can read tensors by computed offsets (no parsing needed at runtime).

```
offset  size  contents
------  ----  --------
0       4     magic 'WOZF'
4       1     version (1)
5       1     vocab_size (32)
6       1     embed_dim  (8)
7       1     block_size (16)
8       1     num_heads  (1)
9       1     n_layers   (1)
10      2     reserved

12      ...   tensors, in fixed order, each as:
              [4 bytes float32 scale][int8 weights, row-major]
```

Tensor order (fixed; encoded in the 6502 firmware):
1. `token_embed.weight`  (vocab × embed)
2. `pos_embed.weight`    (block × embed)
3. `blocks.0.attn.heads.0.key.weight`   (head × embed)
4. `blocks.0.attn.heads.0.query.weight` (head × embed)
5. `blocks.0.attn.heads.0.value.weight` (head × embed)
6. `blocks.0.attn.proj.weight` + bias
7. `blocks.0.mlp.net.0.weight` + bias
8. `blocks.0.mlp.net.2.weight` + bias
9. `lm_head.weight` + bias

LayerNorm γ/β are written in **fp32** (8 floats each × 3 LNs = small).

### Size budget check

Will count it after we write — should land well under 8 KB.


In [ ]:
export_dir = Path('../export'); export_dir.mkdir(exist_ok=True)
out_path = export_dir / 'wozformer.bin'

TENSOR_ORDER = [
    'token_embed.weight',
    'pos_embed.weight',
    'blocks.0.attn.heads.0.key.weight',
    'blocks.0.attn.heads.0.query.weight',
    'blocks.0.attn.heads.0.value.weight',
    'blocks.0.attn.proj.weight',
    'blocks.0.attn.proj.bias',
    'blocks.0.mlp.net.0.weight',
    'blocks.0.mlp.net.0.bias',
    'blocks.0.mlp.net.2.weight',
    'blocks.0.mlp.net.2.bias',
    'lm_head.weight',
    'lm_head.bias',
]
LN_TENSORS = [
    'blocks.0.ln1.weight', 'blocks.0.ln1.bias',
    'blocks.0.ln2.weight', 'blocks.0.ln2.bias',
    'ln_final.weight',     'ln_final.bias',
]

state = dict(model.state_dict())

buf = bytearray()
buf += b'WOZF'
buf += bytes([1, VOCAB_SIZE, EMBED_DIM, BLOCK_SIZE, NUM_HEADS, N_LAYERS, 0, 0])

# int8 tensors
for name in TENSOR_ORDER:
    t = state[name]
    qt, s = quantize_symmetric(t)
    buf += struct.pack('<f', s)
    buf += qt.cpu().numpy().tobytes()

# fp32 LayerNorm params
for name in LN_TENSORS:
    t = state[name].cpu().numpy().astype(np.float32)
    buf += t.tobytes()

out_path.write_bytes(buf)
print(f'wrote {out_path}  ({len(buf):,} bytes)')
print(f'EEPROM budget: 8192 bytes  ({100*len(buf)/8192:.1f}% used)')


## Cell 6 — Cycles-per-token estimate

Verify the model fits the **time** budget too: at 1 MHz, we have ~150 ms per token = 150,000 cycles.

### Multiply count per token

For one forward pass of `T=16` tokens at `embed_dim=8`, the dominant ops per layer:

- **Embedding lookup**: 16 entries × 8 dims = 128 copies (no mults).
- **Q, K, V projections**: `T × embed_dim × head_size = 16 × 8 × 8` each × 3 = **3,072 mults**.
- **Q @ K^T**: `T × T × head_size = 16 × 16 × 8` = **2,048 mults**.
- **(softmax @ V)**: `T × T × head_size = 16 × 16 × 8` = **2,048 mults**.
- **Attention output projection**: `T × embed_dim × embed_dim = 16 × 8 × 8` = **1,024 mults**.
- **MLP up**: `T × embed_dim × 4*embed_dim = 16 × 8 × 32` = **4,096 mults**.
- **MLP down**: `T × 4*embed_dim × embed_dim = 16 × 32 × 8` = **4,096 mults**.
- **lm_head**: `T × embed_dim × vocab = 16 × 8 × 32` = **4,096 mults**.

**Total ≈ 20,500 mults for the entire 16-token forward pass.**

But — and this is the point — for *autoregressive generation* we only need logits at the **last position**. With a KV cache (we'll build this in the 6502 firmware), per-token cost drops to ~1,500 mults. That's the number quoted in the README.

### 6502 multiply cost

The 6502 has no multiply instruction. An 8×8 → 16 unsigned mul takes ~80 cycles via shift-and-add. 1,500 mults × 80 cycles = **120,000 cycles** = 120 ms. Within budget (with optimization headroom).

If we needed more headroom, the 4× MLP expansion is the biggest knob (it's >40% of compute). We could drop it to 2× and lose some quality.


In [ ]:
# Compute the actual number
T = BLOCK_SIZE
ed = EMBED_DIM
hs = ed  # single head
V = VOCAB_SIZE
mlp_mult = 4

full_pass = (
    3 * T * ed * hs +    # Q, K, V projections
    T * T * hs +         # QK^T
    T * T * hs +         # softmax @ V
    T * ed * ed +        # attn output projection
    T * ed * (mlp_mult*ed) +    # MLP up
    T * (mlp_mult*ed) * ed +    # MLP down
    T * ed * V           # lm_head
)

# Per-token (KV-cached, only the last position computed)
per_token = (
    3 * 1 * ed * hs +
    1 * T * hs +
    1 * T * hs +
    1 * ed * ed +
    1 * ed * (mlp_mult*ed) +
    1 * (mlp_mult*ed) * ed +
    1 * ed * V
)

print(f'full T={T} pass:        {full_pass:>6,} mults')
print(f'KV-cached per token:    {per_token:>6,} mults')
print(f'@~80 cycles/mul on 6502: {per_token*80:>7,} cycles = {per_token*80/1e6*1000:.1f} ms @ 1 MHz')
print('budget: ~150,000 cycles = ~150 ms per token')


## Cell 7 — Save a Python-readable checkpoint too

Keep the fp32 checkpoint and quantization table as a `.pt` so the C reference (`reference/`) can load them via a Python helper script. The `.bin` is for the 6502; this is for the development pipeline.


In [ ]:
ckpt = Path('../export/wozformer_quantized.pt')
torch.save({
    'config': dict(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, block_size=BLOCK_SIZE,
                   num_heads=NUM_HEADS, n_layers=N_LAYERS),
    'model_state_fp32_simulated_int8': model.state_dict(),
    'quant_table': {k: (v[0].cpu().numpy().tolist(), v[1]) for k, v in quant_table.items()},
    'itos': itos,
    'val_loss_fp32': val_fp32,
    'val_loss_int8': val_int8,
}, ckpt)
print(f'wrote {ckpt}  ({ckpt.stat().st_size:,} bytes)')


## Post-mortem — we have a model the 6502 can actually run

### What you produced

- `export/wozformer.bin` — int8 weights + fp32 LayerNorm params, packed in a fixed layout. This is what gets burned to the AT28C64 EEPROM.
- `export/wozformer_quantized.pt` — Python-friendly checkpoint for the C reference and any further analysis.
- A working PTQ pipeline you can re-run after retraining (if you change vocab, corpus, etc.).

### What you learned across notebooks 01-07

You can now, from scratch:

- Build a character-level language model.
- Derive scaled dot-product attention from first principles.
- Stack heads, add residuals + LayerNorm + MLP into a pre-norm block.
- Train a transformer end-to-end with PyTorch.
- Quantize the result to int8 with measurable loss.
- Reason about hardware-bound deployment (memory, cycles, accumulator widths).

That's a complete VSLM lifecycle. The architecture you implemented is structurally identical to GPT-2/Llama — yours just has 832 parameters instead of 100 billion.

### Next: leaving PyTorch

The Python work is done. Next steps:

- **`reference/`**: int8 forward pass in plain C. Compile it for both your laptop (sanity-check correctness against this notebook) and Arduino (the known-good reference for byte-by-byte comparison against the 6502).
- **`firmware/6502/`**: ca65 assembly inference. Implement int8 matrix multiply, softmax via lookup table, layernorm in int16, RAM-resident KV cache.
- **`firmware/arduino/`**: EEPROM programmer (writes `wozformer.bin` to AT28C64) + I/O coprocessor for the 16×2 LCD.
- **`hardware/`**: 6502 + SRAM + EEPROM + Arduino, breadboarded, wires everywhere.

Welcome to the hardest part of the project. 
